# Milestone 2: Transformers, Attention, Zero-Shot & Sentence Embeddings
Solves each task step-by-step. Run cells in order — each one prints the answer for that question.

**Note:** Your actual numeric answers will depend on the contents of your `train.csv`. Run every cell and read off the printed value; don't assume the sample numbers from the assignment sheet apply to your data.

In [ ]:
!pip install -q transformers datasets sentence-transformers torch scikit-learn


In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd


## 1. Load train.csv with `datasets` (not pandas) and build `combined_text`

In [ ]:
raw = load_dataset('csv', data_files='train.csv')['train']
print(raw)
print(raw.column_names)


In [ ]:
def add_combined(example):
    example['combined_text'] = example['prompt'] + ' ' + example['A']
    return example

raw = raw.map(add_combined)
print(raw[51]['combined_text'])
print("Length at index 51:", len(raw[51]['combined_text']))


## 2. bert-base-uncased tokenizer: vocab size and [SEP] id

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print("Vocab size:", tokenizer.vocab_size)
print("[SEP] token id:", tokenizer.sep_token_id)


## 3. Tokenize entire prompt column, check input_ids shape

In [ ]:
tokenized = tokenizer(
    raw['prompt'],
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)
print("input_ids shape:", tokenized['input_ids'].shape)


## 4. Attention head dimensionality

In [ ]:
hidden_size = 768
num_heads = 12
head_dim = hidden_size // num_heads
print("Each attention head dimensionality:", head_dim)


## 5. Load bert-base-uncased model, get last_hidden_state shape for row 0

In [ ]:
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = raw[0]['prompt']
inputs0 = tokenizer(row0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs0 = model(**inputs0)

last_hidden_state = outputs0.last_hidden_state
print("last_hidden_state shape:", last_hidden_state.shape)


In [ ]:
cls_vec = last_hidden_state[0, 0, :]
sum_first5 = cls_vec[:5].sum().item()
print("Sum of first 5 CLS values (rounded):", round(sum_first5, 4))


## 6. Attention weight from [CLS] to 'fusion' — last layer, head 0

In [ ]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs_attn = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

attentions = outputs_attn.attentions
last_layer_attn = attentions[-1]
head0_attn = last_layer_attn[0, 0]

# Find token index for 'fusion'
input_ids = inputs_attn['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("Tokens:", tokens)

fusion_idx = tokens.index('fusion')
print("Token index for 'fusion':", fusion_idx)

cls_to_fusion = head0_attn[0, fusion_idx].item()
print("Attention weight CLS -> fusion (rounded):", round(cls_to_fusion, 4))


## 7. Sentence-Transformers cosine similarity (prompt vs Option B, row 0)

In [ ]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompt0 = raw[0]['prompt']
optionB0 = raw[0]['B']

emb_prompt = st_model.encode(prompt0, convert_to_tensor=True)
emb_B = st_model.encode(optionB0, convert_to_tensor=True)

cos_score = util.cos_sim(emb_prompt, emb_B).item()
print("Cosine similarity (rounded):", round(cos_score, 4))


## 8. Build two ranking pipelines: TF-IDF vs MiniLM, compute MAP@3

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = raw.to_pandas()
option_cols = ['A', 'B', 'C', 'D', 'E']


def map_at_3(ranked_lists, correct_answers):
    scores = []
    for ranked, correct in zip(ranked_lists, correct_answers):
        if correct in ranked:
            rank = ranked.index(correct) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return np.mean(scores)


tfidf_top3_all = []
for _, row in df.iterrows():
    options = [row[c] for c in option_cols]
    vec = TfidfVectorizer().fit([row['prompt']] + options)
    prompt_vec = vec.transform([row['prompt']])
    option_vecs = vec.transform(options)
    sims = cosine_similarity(prompt_vec, option_vecs)[0]
    ranked_labels = [option_cols[i] for i in np.argsort(sims)[::-1][:3]]
    tfidf_top3_all.append(ranked_labels)


minilm_top3_all = []
for _, row in df.iterrows():
    options = [row[c] for c in option_cols]
    prompt_emb = st_model.encode(row['prompt'], convert_to_tensor=True)
    option_embs = st_model.encode(options, convert_to_tensor=True)
    sims = util.cos_sim(prompt_emb, option_embs)[0].cpu().numpy()
    ranked_labels = [option_cols[i] for i in np.argsort(sims)[::-1][:3]]
    minilm_top3_all.append(ranked_labels)

correct_answers = df['answer'].tolist()

map3_minilm = map_at_3(minilm_top3_all, correct_answers)
print("MAP@3 (MiniLM):", round(map3_minilm, 4))


count = 0
for tfidf_top3, minilm_top3, correct in zip(tfidf_top3_all, minilm_top3_all, correct_answers):
    if correct not in tfidf_top3 and correct in minilm_top3:
        count += 1

print("Count (correct only in MiniLM top3, not TF-IDF top3):", count)


## 9. Zero-shot classification (facebook/bart-large-mnli)

In [ ]:
zs_classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

row1 = raw[1]
candidate_labels = [row1['A'], row1['B'], row1['C']]

result_softmax = zs_classifier(row1['prompt'], candidate_labels)
print(result_softmax)

top_score = result_softmax['scores'][0]
print("Top-ranked option probability (softmax, rounded):", round(top_score, 4))


In [ ]:
result_sigmoid = zs_classifier(row1['prompt'], candidate_labels, multi_label=True)
print(result_sigmoid)

sum_softmax = sum(result_softmax['scores'][:3])
sum_sigmoid = sum(result_sigmoid['scores'][:3])

diff = abs(sum_softmax - sum_sigmoid)
print("Sum (softmax):", sum_softmax)
print("Sum (sigmoid):", sum_sigmoid)
print("Absolute difference (rounded):", round(diff, 4))


## 10. Generative QA with flan-t5-small

In [ ]:
gen_pipe = pipeline('text2text-generation', model='google/flan-t5-small')

row0 = raw[0]
query = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)
print("Input string:\n", query)

output = gen_pipe(query, max_new_tokens=5)
print("\nModel output:", output[0]['generated_text'])
